# Chapitre 3 - Exercices 3.16 à 3.20

Ce notebook traite cinq exercices de la fin du chapitre 3 (*Linear Methods for
Regression*) :

| Exercice | Thème |
|---|---|
| 3.16 | Formules explicites du tableau 3.4 (cas orthonormal) |
| 3.17 | Le tableau 3.3 refait sur les données *spam* |
| 3.18 | Gradient conjugué et moindres carrés partiels (PLS) |
| 3.19 | Croissance de la norme des coefficients quand on régularise moins |
| 3.20 | Analyse des corrélations canoniques et SVD généralisée |

Les fonctions réutilisables (ridge, lasso, LAR, PCR, PLS, gradient conjugué, CCA…)
sont implémentées explicitement dans le module `esl/ch03_regression_complements.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch03_regression_complements as rc

set_style()
np.set_printoptions(precision=4, suppress=True)

## Notations communes

Dans tout le notebook, $\mathbf X$ désigne la matrice $N\times p$ des entrées
(une ligne par observation), $\mathbf y\in\mathbb R^N$ le vecteur des réponses, et
$\hat\beta = (\mathbf X^T\mathbf X)^{-1}\mathbf X^T\mathbf y$ l'estimateur des
moindres carrés (éq. (3.6) du livre). Le produit scalaire usuel de $\mathbb R^N$ est
noté $\langle \mathbf a, \mathbf b\rangle = \mathbf a^T\mathbf b$.

Sauf mention contraire, les colonnes de $\mathbf X$ sont **centrées** : l'ordonnée à
l'origine est alors estimée par $\bar y$ et peut être oubliée (exercice 3.5).

## Exercice 3.16 : Les estimateurs du tableau 3.4

**Énoncé (reformulé).** On suppose que les colonnes de $\mathbf X$ sont
**orthonormées**, c'est-à-dire $\mathbf X^T\mathbf X = \mathbf I_p$. Établir les formules
du tableau 3.4 du livre, qui donnent explicitement, en fonction des coefficients des
moindres carrés $\hat\beta_j$ :

- le **meilleur sous-ensemble** de taille $M$ : $\hat\beta_j\cdot \mathbb 1\big(|\hat\beta_j|\ge|\hat\beta_{(M)}|\big)$,
  où $|\hat\beta_{(M)}|$ est la $M$-ième plus grande valeur absolue ;
- la **ridge** : $\hat\beta_j/(1+\lambda)$ ;
- le **lasso** : $\operatorname{sign}(\hat\beta_j)\,(|\hat\beta_j|-\lambda)_+$, où $x_+=\max(x,0)$.

### Un lemme clé : la décomposition de la somme des carrés résiduelle

Quand $\mathbf X^T\mathbf X=\mathbf I$, on a $\hat\beta=\mathbf X^T\mathbf y$. Pour tout
$\beta\in\mathbb R^p$, on écrit $\mathbf y-\mathbf X\beta = (\mathbf y-\mathbf X\hat\beta) + \mathbf X(\hat\beta-\beta)$
et on développe le carré de la norme :

$$
\|\mathbf y-\mathbf X\beta\|^2 = \|\mathbf y-\mathbf X\hat\beta\|^2
+ 2(\hat\beta-\beta)^T\mathbf X^T(\mathbf y-\mathbf X\hat\beta)
+ (\hat\beta-\beta)^T\mathbf X^T\mathbf X(\hat\beta-\beta).
$$

- Le terme croisé est nul car $\mathbf X^T(\mathbf y-\mathbf X\hat\beta) = \mathbf X^T\mathbf y-\hat\beta = 0$
  (équations normales).
- Le dernier terme vaut $\|\hat\beta-\beta\|^2$ car $\mathbf X^T\mathbf X=\mathbf I$.

Donc

$$
\boxed{\;\|\mathbf y-\mathbf X\beta\|^2 = \mathrm{RSS}(\hat\beta) + \sum_{j=1}^p(\beta_j-\hat\beta_j)^2.\;}
$$

> **Intuition.** Dans le cas orthonormal, ajuster $\mathbf X\beta$ à $\mathbf y$ revient
> à approcher le point $\hat\beta$ de $\mathbb R^p$ : le problème se **découple**
> coordonnée par coordonnée.

### Meilleur sous-ensemble

Fixons un ensemble $S$ de $M$ indices et imposons $\beta_j=0$ pour $j\notin S$. D'après
le lemme, on minimise
$\sum_{j\in S}(\beta_j-\hat\beta_j)^2 + \sum_{j\notin S}\hat\beta_j^2$ :
le premier terme est rendu nul en prenant $\beta_j=\hat\beta_j$ sur $S$, et il reste
$\sum_{j\notin S}\hat\beta_j^2$.

Il faut ensuite choisir $S$ pour que cette somme soit **minimale**, c'est-à-dire que les
coefficients *exclus* soient les plus petits en valeur absolue : $S$ contient les $M$
plus grandes valeurs $|\hat\beta_j|$. D'où
$\hat\beta_j^{\text{subset}} = \hat\beta_j\,\mathbb 1\big(|\hat\beta_j|\ge|\hat\beta_{(M)}|\big)$
(**seuillage dur**).

### Ridge

La formule (3.44) du livre donne $\hat\beta^{\text{ridge}} = (\mathbf X^T\mathbf X+\lambda\mathbf I)^{-1}\mathbf X^T\mathbf y$.
Avec $\mathbf X^T\mathbf X=\mathbf I$ :

$$
\hat\beta^{\text{ridge}} = \big((1+\lambda)\mathbf I\big)^{-1}\hat\beta = \frac{\hat\beta}{1+\lambda}.
$$

Chaque coefficient est **contracté proportionnellement** : aucun n'est annulé.

### Lasso

Le lasso sous forme lagrangienne (éq. (3.52)) minimise
$\tfrac12\|\mathbf y-\mathbf X\beta\|^2+\lambda\sum_j|\beta_j|$. Grâce au lemme, c'est
(à une constante près)

$$
\sum_{j=1}^p\Big[\tfrac12(\beta_j-\hat\beta_j)^2+\lambda|\beta_j|\Big],
$$

une somme de fonctions d'**une seule** variable : on minimise chaque terme séparément.
Étudions $g(b) = \tfrac12(b-z)^2+\lambda|b|$ avec $z=\hat\beta_j$ et $\lambda>0$.
La fonction $g$ est continue, strictement convexe, tend vers $+\infty$ : elle a un unique
minimiseur. Elle est dérivable partout sauf en $0$.

- **Sur $]0,+\infty[$**, $g'(b) = b-z+\lambda$ s'annule en $b=z-\lambda$, qui est dans
  le domaine si et seulement si $z>\lambda$.
- **Sur $]-\infty,0[$**, $g'(b)=b-z-\lambda$ s'annule en $b=z+\lambda$, admissible si et
  seulement si $z<-\lambda$.
- **Si $|z|\le\lambda$**, $g$ est décroissante sur $]-\infty,0]$ (car $g'(b)=b-z-\lambda\le -z-\lambda\le0$)
  et croissante sur $[0,+\infty[$ (car $g'(b)=b-z+\lambda\ge\lambda-z\ge0$) : le minimum est en $b=0$.

Les trois cas se résument en
$\hat\beta_j^{\text{lasso}} = \operatorname{sign}(\hat\beta_j)\,(|\hat\beta_j|-\lambda)_+$
(**seuillage doux**).

> **Remarque.** Avec la forme sans facteur $\tfrac12$, $\|\mathbf y-\mathbf X\beta\|^2+\lambda\|\beta\|_1$,
> le seuil devient $\lambda/2$ ; la forme du tableau correspond donc à la
> convention (3.52) du livre.

### Vérification numérique

On construit un plan orthonormé ($\mathbf X = $ facteur $Q$ d'une décomposition QR),
puis on compare les formules fermées aux solutions obtenues par des algorithmes
génériques : ridge par résolution du système, lasso par descente de coordonnées, meilleur
sous-ensemble par recherche exhaustive.

In [ ]:
rng = np.random.RandomState(0)
N, p = 50, 6
Xo = np.linalg.qr(rng.randn(N, p))[0]          # colonnes orthonormées
yo = Xo @ np.array([3.0, -2.0, 1.0, 0.5, -0.3, 0.0]) + 0.5 * rng.randn(N)
beta_ls = Xo.T @ yo
print("X^T X = I :", np.allclose(Xo.T @ Xo, np.eye(p)))
print("beta LS     :", beta_ls)